In [21]:
import numpy as np
import torch
import torch.nn as nn

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error

from torch.utils.data import TensorDataset
from torch.utils.data import DataLoader

In [22]:
data = np.load("pems08.npz")["data"]

traffic = data[:, :, 0]

print(traffic.shape)

(17856, 170)


In [23]:
scaler = MinMaxScaler()

traffic_scaled = scaler.fit_transform(
    traffic
)

print(
    traffic_scaled.shape
)

(17856, 170)


In [24]:
X = []
y = []

sequence_length = 12

for i in range(
    len(traffic_scaled)
    -
    sequence_length
):

    X.append(
        traffic_scaled[
            i:i+sequence_length
        ]
    )

    y.append(
        traffic_scaled[
            i+sequence_length
        ]
    )

X = np.array(X)
y = np.array(y)

print(X.shape)
print(y.shape)

(17844, 12, 170)
(17844, 170)


In [25]:
split = int(
    len(X) * 0.8
)

X_train = X[:split]
X_test = X[split:]

y_train = y[:split]
y_test = y[split:]

print(X_train.shape)
print(X_test.shape)

(14275, 12, 170)
(3569, 12, 170)


In [26]:
X_train = torch.tensor(
    X_train,
    dtype=torch.float32
)

X_test = torch.tensor(
    X_test,
    dtype=torch.float32
)

y_train = torch.tensor(
    y_train,
    dtype=torch.float32
)

y_test = torch.tensor(
    y_test,
    dtype=torch.float32
)

In [27]:
train_dataset = TensorDataset(
    X_train,
    y_train
)

train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True
)

In [28]:
class GraphConv(nn.Module):

    def __init__(
        self,
        num_nodes,
        in_features,
        out_features
    ):

        super().__init__()

        self.node_embeddings = nn.Parameter(
            torch.randn(
                num_nodes,
                16
            )
        )

        self.linear = nn.Linear(
            in_features,
            out_features
        )

    def forward(self,x):

        adj = torch.softmax(
            torch.relu(
                self.node_embeddings
                @
                self.node_embeddings.T
            ),
            dim=1
        )

        x = torch.einsum(
            "ij,btj->bti",
            adj,
            x
        )

        x = self.linear(x)

        return torch.relu(x)

In [29]:
class GCNLSTM(nn.Module):

    def __init__(self):

        super().__init__()

        self.gcn = GraphConv(
            num_nodes=170,
            in_features=170,
            out_features=170
        )

        self.lstm = nn.LSTM(
            input_size=170,
            hidden_size=64,
            batch_first=True
        )

        self.fc = nn.Linear(
            64,
            170
        )

    def forward(self,x):

        x = self.gcn(x)

        output, (hidden, cell) = self.lstm(x)

        hidden = hidden[-1]

        x = self.fc(hidden)

        return x

In [30]:
model = GCNLSTM()

X_batch, y_batch = next(iter(train_loader))

pred = model(X_batch)

print(pred.shape)
print(y_batch.shape)

torch.Size([64, 170])
torch.Size([64, 170])


In [31]:
model = GCNLSTM()

criterion = nn.MSELoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

import time

train_start = time.time()

epochs = 30

for epoch in range(epochs):

    model.train()

    total_loss = 0

    for X_batch, y_batch in train_loader:

        optimizer.zero_grad()

        pred = model(X_batch)

        loss = criterion(
            pred,
            y_batch
        )

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    print(
        f"Epoch {epoch+1}/{epochs}",
        f"Loss: {total_loss/len(train_loader):.6f}"
    )

Epoch 1/30 Loss: 0.021747
Epoch 2/30 Loss: 0.006321
Epoch 3/30 Loss: 0.005123
Epoch 4/30 Loss: 0.004531
Epoch 5/30 Loss: 0.004170
Epoch 6/30 Loss: 0.003908
Epoch 7/30 Loss: 0.003680
Epoch 8/30 Loss: 0.003483
Epoch 9/30 Loss: 0.003339
Epoch 10/30 Loss: 0.003225
Epoch 11/30 Loss: 0.003131
Epoch 12/30 Loss: 0.003047
Epoch 13/30 Loss: 0.002966
Epoch 14/30 Loss: 0.002904
Epoch 15/30 Loss: 0.002849
Epoch 16/30 Loss: 0.002799
Epoch 17/30 Loss: 0.002767
Epoch 18/30 Loss: 0.002746
Epoch 19/30 Loss: 0.002718
Epoch 20/30 Loss: 0.002661
Epoch 21/30 Loss: 0.002659
Epoch 22/30 Loss: 0.002634
Epoch 23/30 Loss: 0.002617
Epoch 24/30 Loss: 0.002583
Epoch 25/30 Loss: 0.002552
Epoch 26/30 Loss: 0.002539
Epoch 27/30 Loss: 0.002527
Epoch 28/30 Loss: 0.002509
Epoch 29/30 Loss: 0.002494
Epoch 30/30 Loss: 0.002495


In [32]:
train_time = time.time() - train_start

print("Time Taken:", train_time)

Time Taken: 156.4322738647461


In [33]:
torch.save(
    model.state_dict(),
    "GCN-LSTM-PEMS08.pth"
)

In [34]:
model.eval()


infer_start = time.time()


with torch.no_grad():

    predictions = model(
        X_test
    )

predictions = predictions.numpy()

infer_time = time.time() - infer_start
print("Infer Time:", infer_time)

true_values = y_test.numpy()

mae = mean_absolute_error(
    true_values,
    predictions
)

rmse = np.sqrt(
    mean_squared_error(
        true_values,
        predictions
    )
)

print("MAE:", mae)
print("RMSE:", rmse)

Infer Time: 0.2944164276123047
MAE: 0.037673753
RMSE: 0.05577659


In [35]:
from sklearn.metrics import r2_score

mape = np.mean(
    np.abs((true_values - predictions) /
           np.maximum(np.abs(true_values), 1e-6))
) * 100

r2 = r2_score(
    true_values.flatten(),
    predictions.flatten()
)
print("MAPE:", mape)
print("R2:", r2)

MAPE: 18018.9453125
R2: 0.9497677371583582


In [36]:
params = sum(
    p.numel()
    for p in model.parameters()
)

print("Parameters:", params)

Parameters: 103256
